In [ ]:
from datetime import timedelta
import io
from PIL import Image
from cairosvg import svg2png
import numpy as np
from glider import GoProVideoMetadata, IGCFile
import cv2

In [ ]:
video_filename = 'GX010107.MP4'
igc_data = IGCFile("2026-05-01-XTR-A68A17562166-01.IGC")

gopro_meta = GoProVideoMetadata.from_video(video_filename)
gopro_meta.creation_time -= timedelta(hours=2)

In [ ]:
# Compass Data
svg_data = open("compass-north-svgrepo-com.svg", "rb").read()
png_data = svg2png(bytestring=svg_data, output_width=100)
overlay = Image.open(io.BytesIO(png_data)).convert("RGBA")
overlay_center = [x/2 for x in overlay.size]

In [ ]:
start_datetime = gopro_meta.creation_time
cap = cv2.VideoCapture(video_filename)

fps = cap.get(cv2.CAP_PROP_FPS)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')#(*'mp4v')
out = cv2.VideoWriter('output_with_alt.mp4', fourcc, fps, (width, height))
frame_count = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    if frame_count < 41000:
        frame_count += 1
        continue

    current_frame_time = start_datetime + timedelta(seconds=frame_count/fps)
    heading_value = igc_data.get_bearing_at_time(current_frame_time)

    text = f"ALT {igc_data.get_altitude_at_time(current_frame_time)} m"
    cv2.putText(frame, text, (50, height - 50), cv2.FONT_ITALIC, 1.5, (0, 255, 0), 3)
    img = Image.fromarray(frame)
    rotated_overlay = overlay.rotate(-heading_value.bearing_deg, center=overlay_center, resample=Image.BICUBIC)
    img.paste(rotated_overlay, (100, 100), rotated_overlay)
    frame = np.array(img)
    out.write(frame)
    frame_count += 1

cap.release()
out.release()
print("Export Complete!")